In [10]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme()

# "Public thinks" = YouGov, 9–14 Nov 2025.  "Reality" = IFS.  rank 1 = most spent.
perceived_order = ["Debt interest","NHS","Working age benefits","Public order & safety","Defence",
                   "Pensioners","Education","Social care","Overseas aid","Housing","Transport"]
actual_order    = ["NHS","Working age benefits","Pensioners","Education","Debt interest","Defence",
                   "Public order & safety","Transport","Social care","Housing","Overseas aid"]

HIGHLIGHT = "Debt interest"
ACCENT, INK, MUTED = "#e6224b", "#4a4f60", "#9aa1b0"

records = []
for order, side, xpos in [(perceived_order,"Public thinks",0), (actual_order,"Reality",1)]:
    for position, area in enumerate(order, start=1):
        records.append({"area":area, "side":side, "rank":position, "x":xpos, "hl":area==HIGHLIGHT})
df = pd.DataFrame(records)

x = alt.X("x:Q", scale=alt.Scale(domain=[-0.15, 1.15]), axis=None)
y = alt.Y("rank:Q", scale=alt.Scale(reverse=True, domain=[-0.4, 11.6]), axis=None)

def labels(side, align, dx):
    rows = df[df["side"] == side]
    other  = alt.Chart(rows[~rows["hl"]]).mark_text(
        align=align, dx=dx, fontSize=13, color=MUTED).encode(x=x, y=y, text="area:N")
    strong = alt.Chart(rows[rows["hl"]]).mark_text(
        align=align, dx=dx, fontSize=13, fontWeight="bold", color=ACCENT).encode(x=x, y=y, text="area:N")
    return other + strong

left, right = labels("Public thinks","right",-10), labels("Reality","left",10)

connector = alt.Chart(df[df["hl"]]).mark_line(color=ACCENT, strokeWidth=2.5).encode(x=x, y=y, detail="area:N")
dots      = alt.Chart(df[df["hl"]]).mark_circle(color=ACCENT, size=90).encode(x=x, y=y)

head_left  = alt.Chart(pd.DataFrame([{"x":0,"rank":0}])).mark_text(
    text="PUBLIC THINKS", align="right", dx=-10, fontSize=11, fontWeight="bold", color=INK).encode(x=x, y=y)
head_right = alt.Chart(pd.DataFrame([{"x":1,"rank":0}])).mark_text(
    text="REALITY", align="left", dx=10, fontSize=11, fontWeight="bold", color=INK).encode(x=x, y=y)

chart = (head_left + head_right + left + right + connector + dots).properties(
    width=300, height=470,
    padding={"left":50, "right":50, "top":5, "bottom":5}
).configure_view(strokeWidth=0)

styles.save(chart, name="perception_vs_reality", svg=True)
chart

alt.LayerChart(...)